# NPY Cache Viewer

Visualise samples from the preprocessed NPY cache produced by `scripts/preprocess_cache.py`
(`train.npy` / `val.npy` / `meta.json`, memory-mapped RAW snippets).

Shows both **raw** snippets (as stored in the cache) and **preprocessed** snippets
(after `bandpass_correct` + `core_transform`, i.e. what the model sees).

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..").resolve()))
from src.data.preprocessing import bandpass_correct, core_transform

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Load the NPY cache (memory-mapped)

In [ ]:
CACHE_DIR = Path("../data/processed/cache_gbt_fine_exotica")

train = np.load(str(CACHE_DIR / "train.npy"), mmap_mode="r")
val = np.load(str(CACHE_DIR / "val.npy"), mmap_mode="r")

n_obs, tchans_per_obs, fchans = train.shape[1:]

print(f"Train: {train.shape}  ({train.nbytes / 1e9:.2f} GB)")
print(f"Val  : {val.shape}  ({val.nbytes / 1e9:.2f} GB)")
print(f"n_obs={n_obs}  tchans_per_obs={tchans_per_obs}  fchans={fchans}")

meta_path = CACHE_DIR / "meta.json"
meta = json.loads(meta_path.read_text())
print(f"\nMetadata: {json.dumps(meta, indent=2)}")

## 2. Helper: preprocess a single snippet

In [ ]:
PREPROC = meta["preprocessing"]  # {"bandpass_method", "poly_degree", "mad_epsilon"}

def preprocess(snippet_3d):
    """(n_obs, tchans_per_obs, fchans) -> (tchans, fchans) preprocessed."""
    frame = np.concatenate(snippet_3d, axis=0)
    frame = bandpass_correct(frame, method=PREPROC["bandpass_method"], poly_degree=PREPROC["poly_degree"])
    frame = core_transform(frame, PREPROC["mad_epsilon"])
    return frame

## 3. Random samples: raw vs preprocessed

In [ ]:
N_SHOW = 8
rng = np.random.default_rng(42)
indices = rng.choice(len(train), size=N_SHOW, replace=False)

fig, axes = plt.subplots(N_SHOW, 2, figsize=(14, 3 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(indices):
    raw_4d = train[idx]
    raw_concat = np.concatenate(raw_4d, axis=0)
    processed = preprocess(raw_4d)

    ax_raw = axes[row, 0]
    im0 = ax_raw.imshow(raw_concat, aspect="auto", origin="lower", interpolation="none")
    ax_raw.set_title(f"Sample {idx} — Raw", fontsize=10)
    ax_raw.set_ylabel("Time bin")
    plt.colorbar(im0, ax=ax_raw, fraction=0.03)

    ax_proc = axes[row, 1]
    vmin, vmax = np.percentile(processed, [2, 98])
    im1 = ax_proc.imshow(processed, aspect="auto", origin="lower", interpolation="none",
                         vmin=vmin, vmax=vmax)
    ax_proc.set_title(f"Sample {idx} — Preprocessed", fontsize=10)
    plt.colorbar(im1, ax=ax_proc, fraction=0.03)

axes[-1, 0].set_xlabel("Freq channel")
axes[-1, 1].set_xlabel("Freq channel")
fig.suptitle("Train split — Random samples", fontsize=13, y=1.0)
plt.tight_layout()
plt.show()

## 4. Per-observation breakdown

Show each observation within a cadence separately (useful for spotting ON/OFF structure).

In [ ]:
idx = rng.choice(len(train))
sample = train[idx]  # (n_obs, tchans_per_obs, fchans)

fig, axes = plt.subplots(1, n_obs, figsize=(3 * n_obs, 3), sharey=True)
if n_obs == 1:
    axes = [axes]
for oi in range(n_obs):
    im = axes[oi].imshow(sample[oi], aspect="auto", origin="lower", interpolation="none")
    axes[oi].set_title(f"Obs {oi}", fontsize=9)
    axes[oi].set_xlabel("Freq channel")
    plt.colorbar(im, ax=axes[oi], fraction=0.05)
axes[0].set_ylabel("Time bin")
fig.suptitle(f"Sample {idx} — per-observation (raw)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Pixel distribution (raw vs preprocessed)

In [ ]:
N_HIST = min(500, len(train))
hist_indices = rng.choice(len(train), size=N_HIST, replace=False)

raw_vals = []
proc_vals = []
for i in hist_indices:
    raw_concat = np.concatenate(train[i], axis=0)
    raw_vals.append(raw_concat.ravel())
    proc_vals.append(preprocess(train[i]).ravel())

raw_flat = np.concatenate(raw_vals)
proc_flat = np.concatenate(proc_vals)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(raw_flat, bins=200, density=True, alpha=0.7, color="steelblue")
ax1.set_title("Raw pixel distribution")
ax1.set_xlabel("Value")
ax1.set_ylabel("Density")
ax1.set_yscale("log")

ax2.hist(proc_flat, bins=200, density=True, alpha=0.7, color="coral")
ax2.set_title("Preprocessed pixel distribution")
ax2.set_xlabel("Value")
ax2.set_yscale("log")

plt.tight_layout()
plt.show()

print(f"Raw   — min={raw_flat.min():.2f}  max={raw_flat.max():.2f}  "
      f"mean={raw_flat.mean():.2f}  std={raw_flat.std():.2f}")
print(f"Proc  — min={proc_flat.min():.2f}  max={proc_flat.max():.2f}  "
      f"mean={proc_flat.mean():.4f}  std={proc_flat.std():.4f}")

## 6. Val split samples

In [ ]:
N_VAL_SHOW = 4
val_indices = rng.choice(len(val), size=min(N_VAL_SHOW, len(val)), replace=False)

fig, axes = plt.subplots(len(val_indices), 2, figsize=(14, 3 * len(val_indices)))
if len(val_indices) == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(val_indices):
    raw_concat = np.concatenate(val[idx], axis=0)
    processed = preprocess(val[idx])

    im0 = axes[row, 0].imshow(raw_concat, aspect="auto", origin="lower", interpolation="none")
    axes[row, 0].set_title(f"Val {idx} — Raw", fontsize=10)
    axes[row, 0].set_ylabel("Time bin")
    plt.colorbar(im0, ax=axes[row, 0], fraction=0.03)

    vmin, vmax = np.percentile(processed, [2, 98])
    im1 = axes[row, 1].imshow(processed, aspect="auto", origin="lower", interpolation="none",
                              vmin=vmin, vmax=vmax)
    axes[row, 1].set_title(f"Val {idx} — Preprocessed", fontsize=10)
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.03)

axes[-1, 0].set_xlabel("Freq channel")
axes[-1, 1].set_xlabel("Freq channel")
fig.suptitle("Val split — Random samples", fontsize=13, y=1.0)
plt.tight_layout()
plt.show()

## 7. Reconstruction-error proxy (self-MSE per sample)

Approximate the per-sample "interest" by computing MSE vs the batch mean —
outliers here are the snippets most unlike the average background.

In [ ]:
N_SCORE = min(2000, len(train))
score_idx = rng.choice(len(train), size=N_SCORE, replace=False)

processed_batch = np.stack([preprocess(train[i]) for i in score_idx])
batch_mean = processed_batch.mean(axis=0)
mse = ((processed_batch - batch_mean) ** 2).mean(axis=(1, 2))

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(mse, bins=100, alpha=0.7, color="teal")
ax.axvline(np.percentile(mse, 95), color="red", ls="--", label="95th pct")
ax.axvline(np.percentile(mse, 99), color="darkred", ls="--", label="99th pct")
ax.set_xlabel("MSE vs batch mean")
ax.set_ylabel("Count")
ax.set_title("Per-sample MSE (proxy for anomaly score)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
top_k = 4
top_indices = np.argsort(mse)[-top_k:][::-1]

fig, axes = plt.subplots(top_k, 2, figsize=(14, 3 * top_k))
for row, ti in enumerate(top_indices):
    real_idx = score_idx[ti]
    proc = processed_batch[ti]
    raw_concat = np.concatenate(train[real_idx], axis=0)

    im0 = axes[row, 0].imshow(raw_concat, aspect="auto", origin="lower", interpolation="none")
    axes[row, 0].set_title(f"#{real_idx} MSE={mse[ti]:.3f} — Raw", fontsize=10)
    axes[row, 0].set_ylabel("Time bin")
    plt.colorbar(im0, ax=axes[row, 0], fraction=0.03)

    vmin, vmax = np.percentile(proc, [2, 98])
    im1 = axes[row, 1].imshow(proc, aspect="auto", origin="lower", interpolation="none",
                              vmin=vmin, vmax=vmax)
    axes[row, 1].set_title(f"#{real_idx} MSE={mse[ti]:.3f} — Preprocessed", fontsize=10)
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.03)

axes[-1, 0].set_xlabel("Freq channel")
axes[-1, 1].set_xlabel("Freq channel")
fig.suptitle(f"Top-{top_k} highest MSE (most anomalous)", fontsize=13, y=1.0)
plt.tight_layout()
plt.show()